# Pinch Analysis: Crude Preheat Train
This notebook shows how the pinch, utility targets, and graph shapes move when the minimum approach temperature assumption changes.


## Case context
The packaged `crude_preheat_train.json` case represents a small crude-unit preheat train with named hot and cold duties. `PinchWorkspace` keeps named study cases, while each case remains a real `PinchProblem` with the familiar `target`, `plot`, `summary_frame`, and `compare_to` workflow.


In [47]:
import json
from importlib.resources import files

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from OpenPinch import PinchWorkspace

_CASE_ROOT = files("stream_data")

In [48]:
source = _CASE_ROOT.joinpath("p_Perry et al.json").read_text(encoding="utf-8")
workspace = PinchWorkspace(
    json.loads(source),
    project_name="debug_case",
)
baseline = workspace.use_case("baseline")
summary = baseline.summary_frame()
catalog = baseline.plot.catalog()

{
    "cases": workspace.list_cases(),
    "active_case": workspace.active_case_name,
    "graph_count": len(catalog),
    "zone_tree_present": "zone_tree" in workspace.get_case_payload("baseline"),
}

{'cases': ['baseline'],
 'active_case': 'baseline',
 'graph_count': 37,
 'zone_tree_present': True}

In [49]:
summary

,Target,Hot Utility Target,Cold Utility Target,Heat Recovery,Hot Pinch,Cold Pinch,Hot Utilities,Cold Utilities
0,debug_case/Direct Integration,11411.77,0.0,29597.25,25.00,25.00,"VHPS: 0.00, HPS: 0.00, MPS: 11411.77","HPS_1: 0.00, MPS_1: 0.00, CW: 0.00, CHW: 0.00"
1,debug_case/Total Process Target,17588.77,6177.0,23420.25,NaN,NaN,"VHPS: 0.00, HPS: 0.00, MPS: 17588.77","HPS_1: 0.00, MPS_1: 0.00, CW: 6063.36, CHW: 11..."
2,debug_case/Total Site Target,17588.77,6177.0,23420.25,159.99,25.01,"VHPS: 0.00, HPS: 0.00, MPS: 17588.77","HPS_1: 0.00, MPS_1: 0.00, CW: 6063.36, CHW: 11..."
3,A/Direct Integration,0.00,3977.0,7500.00,160.00,160.00,"VHPS: 0.00, HPS: 0.00, MPS: 0.00","HPS_1: 0.00, MPS_1: 0.00, CW: 3977.00, CHW: 0.00"
4,B/Direct Integration,6200.00,2200.0,15800.00,85.00,85.00,"VHPS: 0.00, HPS: 0.00, MPS: 6200.00","HPS_1: 0.00, MPS_1: 0.00, CW: 2086.36, CHW: 11..."
5,C/Direct Integration,388.77,0.0,120.25,28.00,28.00,"VHPS: 0.00, HPS: 0.00, MPS: 388.77","HPS_1: 0.00, MPS_1: 0.00, CW: 0.00, CHW: 0.00"
6,D/Direct Integration,11000.00,0.0,0.00,25.00,25.00,"VHPS: 0.00, HPS: 0.00, MPS: 11000.00","HPS_1: 0.00, MPS_1: 0.00, CW: 0.00, CHW: 0.00"


## Baseline graphs from the built-in plot accessor
Because each workspace case is a real `PinchProblem`, the notebook can use the normal `plot` accessor directly instead of rebuilding figures from serialized helper payloads.


In [50]:
catalog.loc[
    catalog["Target"] == "B/Direct Integration",
    ["Target", "Graph Type", "Graph Name"],
].reset_index(drop=True)

,Target,Graph Type,Graph Name
0,B/Direct Integration,Composite Curves,Graph 1
1,B/Direct Integration,Shifted Composite Curves,Graph 2
2,B/Direct Integration,Balanced Composite Curves,Graph 3
3,B/Direct Integration,Grand Composite Curve,Graph 4
4,B/Direct Integration,Grand Composite Curve (Real),Graph 5
5,B/Direct Integration,Net Load Profiles,Graph 6
6,B/Direct Integration,Grand Composite Curve with Heat Pump,Graph 7


In [57]:
baseline.plot.composite_curve(
    zone_name="B",
)

In [58]:
baseline.plot.shifted_composite_curve(
    zone_name="B",
)

In [59]:
baseline.plot.grand_composite_curve(
    zone_name="B",
)

## One named case before the sweep
Clone the baseline into a named case, adjust the root `dt_cont_multiplier`, and compare the two `PinchProblem` summaries directly.


In [54]:
wide_dt = workspace.copy_case("baseline", "wide_dt", activate=False)
wide_dt.set_dt_cont_multiplier(2.0)
workspace.compare_cases(
    "baseline",
    "wide_dt",
    target_name="crude_preheat_train/Direct Integration",
)

,Target,Hot Utility Target,Cold Utility Target,Heat Recovery,Hot Pinch,Cold Pinch
baseline,debug_case/Direct Integration,11411.770000,0.000000,29597.250000,25.0,25.0
wide_dt,debug_case/Direct Integration,12546.779091,1135.009091,28462.240909,35.0,35.0
Change,debug_case/Direct Integration,1135.009091,1135.009091,-1135.009091,10.0,10.0


In [55]:
rows = []
for factor in np.linspace(0.5, 10.0, 96):
    case_name = f"dt_mult_{factor:05.2f}".replace(".", "_")
    case = workspace.copy_case("baseline", case_name, activate=False)
    case.set_dt_cont_multiplier(float(factor))
    row = (
        case.summary_frame()
        .loc[lambda frame: frame["Target"] == "B/Direct Integration"]
        .iloc[0]
    )
    rows.append(
        {
            "dt_cont multiplier": factor,
            "Hot Utility Target": row["Hot Utility Target"],
            "Cold Utility Target": row["Cold Utility Target"],
            "Heat Recovery": row["Heat Recovery"],
            "Hot Pinch": row["Hot Pinch"],
            "Cold Pinch": row["Cold Pinch"],
        }
    )

sensitivity = pd.DataFrame(rows)
sensitivity

,dt_cont multiplier,Hot Utility Target,Cold Utility Target,Heat Recovery,Hot Pinch,Cold Pinch
0,0.5,4639.393939,639.393939,1.736061e+04,80.0,80.0
1,0.6,4951.515152,951.515152,1.704848e+04,81.0,81.0
2,0.7,5263.636364,1263.636364,1.673636e+04,82.0,82.0
3,0.8,5575.757576,1575.757576,1.642424e+04,83.0,83.0
4,0.9,5887.878788,1887.878788,1.611212e+04,84.0,84.0
...,...,...,...,...,...,...
91,9.6,22000.000000,18000.000000,7.275958e-12,116.0,104.0
92,9.7,22000.000000,18000.000000,7.275958e-12,117.0,103.0
93,9.8,22000.000000,18000.000000,7.275958e-12,118.0,102.0
94,9.9,22000.000000,18000.000000,7.275958e-12,119.0,101.0


In [56]:
series_colors = {
    "Hot Utility Target": "#c0392b",
    "Cold Utility Target": "#2471a3",
    "Heat Recovery": "#138d75",
    "Hot Pinch": "#d68910",
    "Cold Pinch": "#7d3c98",
}

sensitivity_fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    subplot_titles=(
        "Utility Targets and Heat Recovery",
        "Pinch Temperatures",
    ),
)

for column in ["Hot Utility Target", "Cold Utility Target", "Heat Recovery"]:
    sensitivity_fig.add_trace(
        go.Scatter(
            x=sensitivity["dt_cont multiplier"],
            y=sensitivity[column],
            mode="lines+markers",
            name=column,
            line={"color": series_colors[column], "width": 3},
            marker={"color": series_colors[column], "size": 7},
        ),
        row=1,
        col=1,
    )

for column in ["Hot Pinch", "Cold Pinch"]:
    sensitivity_fig.add_trace(
        go.Scatter(
            x=sensitivity["dt_cont multiplier"],
            y=sensitivity[column],
            mode="lines+markers",
            name=column,
            line={"color": series_colors[column], "width": 3},
            marker={"color": series_colors[column], "size": 7},
        ),
        row=2,
        col=1,
    )

sensitivity_fig.update_xaxes(title_text="dt_cont multiplier", row=2, col=1)
sensitivity_fig.update_yaxes(title_text="kW or degC", row=1, col=1)
sensitivity_fig.update_yaxes(title_text="degC", row=2, col=1)
sensitivity_fig.update_layout(title="dt_cont sensitivity overview")
sensitivity_fig